<a href="https://www.kaggle.com/code/briankimanzi/bean-classification?scriptVersionId=307400066" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 🫘 Bean Classification — YOLOv8x Segmentation
**Kaggle Notebook** | Optimised for F1 score | Run cells top to bottom

## 1. Install Dependencies

In [1]:
!pip install ultralytics gdown -q
!pip install pandas numpy Pillow opencv-python-headless -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.0 MB/s eta 0:00:00a 0:00:01


## 2. Imports

In [2]:
import os, shutil, ast, random, cv2, subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from ultralytics import YOLO
import torch
from sklearn.model_selection import train_test_split
from IPython.display import FileLink, display
from collections import defaultdict
import yaml

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## 3. Discover Input Files

In [3]:
for dirname, dirs, filenames in os.walk('/kaggle/input'):
    print(f"{dirname}  ({len(filenames)} files)")

/kaggle/input  (0 files)
/kaggle/input/datasets  (0 files)
/kaggle/input/datasets/briankimanzi  (0 files)
/kaggle/input/datasets/briankimanzi/beans-clasification-training  (1 files)
/kaggle/input/datasets/briankimanzi/beans-classification-testing  (1 files)


## 4. Set All Paths (edit here if needed)

In [4]:
TRAIN_CSV     = '/kaggle/input/datasets/briankimanzi/beans-clasification-training/Train.csv'
TEST_CSV      = '/kaggle/input/datasets/briankimanzi/beans-classification-testing/Test.csv'
TRAIN_IMG_DIR = '/kaggle/working/images'
TEST_IMG_DIR  = '/kaggle/working/images'
YOLO_DIR      = '/kaggle/working/yolo_dataset'
RUNS_DIR      = '/kaggle/working/runs'
BEST_MODEL    = '/kaggle/working/best.pt'
SUBMISSION    = '/kaggle/working/submission.csv'

print("Paths configured")

Paths configured


## 5. Download Images (skips if already present)

In [5]:
import gdown, zipfile

FILE_ID = "1WJPkaa8lNhE13eVb_mjhcdeano_7tH8R"  # ← your Google Drive file ID
IMG_DIR = '/kaggle/working/images'

if Path(IMG_DIR).exists() and len(list(Path(IMG_DIR).iterdir())) > 100:
    print(f"Images already exist ({len(list(Path(IMG_DIR).iterdir()))} files) — skipping download")
else:
    print("Downloading images.zip from Google Drive...")
    gdown.download(id=FILE_ID, output="/kaggle/working/images.zip", quiet=False, fuzzy=True)
    print("Extracting...")
    with zipfile.ZipFile("/kaggle/working/images.zip", "r") as z:
        z.extractall("/kaggle/working/")
    os.remove("/kaggle/working/images.zip")
    print(f"Done! {len(list(Path(IMG_DIR).iterdir()))} images ready")

Downloading...
From (original): https://drive.google.com/uc?id=1WJPkaa8lNhE13eVb_mjhcdeano_7tH8R
From (redirected): https://drive.google.com/uc?id=1WJPkaa8lNhE13eVb_mjhcdeano_7tH8R&confirm=t&uuid=de07fd5c-0fe6-41f3-b9ef-23f9354498d3
To: /kaggle/working/images.zip
100%|██████████| 6.07G/6.07G [00:36<00:00, 165MB/s] 


Extracting...
Done! 2713 images ready


## 6. Check Disk Space

In [6]:
for item in sorted(os.listdir('/kaggle/working')):
    path = f'/kaggle/working/{item}'
    if os.path.isdir(path):
        size = subprocess.run(['du', '-sh', path], capture_output=True, text=True).stdout.split()[0]
        print(f"{size:<8} {path}")
    else:
        size = os.path.getsize(path) / 1024**2
        print(f"{size:>6.1f} MB  {path}")

result = subprocess.run(['df', '-h', '/kaggle/working'], capture_output=True, text=True)
print()
print(result.stdout)

24K      /kaggle/working/.virtual_documents
5.7G     /kaggle/working/images

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  5.7G   14G  29% /kaggle/working



## 7. Load CSVs & Explore Data

In [7]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("Train shape:", train_df.shape)
print("Test shape: ", test_df.shape)
print()
print("Train columns:", train_df.columns.tolist())
print()
print("Class distribution:")
print(train_df['class'].value_counts())
print()
print("Unique train images:", train_df["Image_ID"].nunique())
print("Unique test images: ", test_df["Image_ID"].nunique())
print()
print("Sample Image_IDs:")
print(train_df['Image_ID'].unique()[:5])

Train shape: (12142, 6)
Test shape:  (352, 3)

Train columns: ['Image_ID', 'image_width', 'image_height', 'bbox', 'points', 'class']

Class distribution:
class
Flower_closed    6003
Flower_open      4085
Plant_bean       2054
Name: count, dtype: int64

Unique train images: 985
Unique test images:  352

Sample Image_IDs:
['zpnCucYXL5Ta' 'RoVsYaebTYkI' 'RQT1orUqN3Ur' 'a0AhtNcl8PU2' 'LggZwvNYR200']


## 8. Build YOLO Dataset

In [8]:
CLASS_MAP = {
    'Plant_bean':    0,
    'Flower_open':   1,
    'Flower_closed': 2
}
ID_TO_CLASS = {v: k for k, v in CLASS_MAP.items()}
CLASS_NAMES = list(CLASS_MAP.keys())

def parse_points(points_str):
    pairs = points_str.strip().split(';')
    coords = []
    for p in pairs:
        x, y = p.strip().split(',')
        coords.append((float(x), float(y)))
    return coords

def normalize_polygon(points, img_w, img_h):
    normalized = []
    for x, y in points:
        normalized.extend([x / img_w, y / img_h])
    return normalized

def create_yolo_dataset(df, img_src_dir, split_name, yolo_dir):
    img_out = Path(yolo_dir) / 'images' / split_name
    lbl_out = Path(yolo_dir) / 'labels' / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    grouped = df.groupby('Image_ID')
    skipped = 0
    for img_id, group in grouped:
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
            candidate = Path(img_src_dir) / f'{img_id}{ext}'
            if candidate.exists():
                img_path = candidate
                break
        if img_path is None:
            skipped += 1
            continue
        dest_img = img_out / img_path.name
        if not dest_img.exists():
            shutil.copy(img_path, dest_img)
        img_w = group['image_width'].iloc[0]
        img_h = group['image_height'].iloc[0]
        label_lines = []
        for _, row in group.iterrows():
            cls_id = CLASS_MAP.get(row['class'])
            if cls_id is None:
                continue
            try:
                points = parse_points(row['points'])
                norm = normalize_polygon(points, img_w, img_h)
                coords_str = ' '.join(f'{v:.6f}' for v in norm)
                label_lines.append(f'{cls_id} {coords_str}')
            except:
                continue
        lbl_file = lbl_out / f'{img_id}.txt'
        with open(lbl_file, 'w') as f:
            f.write('\n'.join(label_lines))
    print(f'[{split_name}] Done. Skipped {skipped} images not found on disk.')

unique_ids = train_df['Image_ID'].unique()
train_ids, val_ids = train_test_split(unique_ids, test_size=0.15, random_state=42)
train_split = train_df[train_df['Image_ID'].isin(train_ids)]
val_split   = train_df[train_df['Image_ID'].isin(val_ids)]

print(f'Train images: {len(train_ids)} | Val images: {len(val_ids)}')

if os.path.exists(YOLO_DIR):
    shutil.rmtree(YOLO_DIR)

create_yolo_dataset(train_split, TRAIN_IMG_DIR, 'train', YOLO_DIR)
create_yolo_dataset(val_split,   TRAIN_IMG_DIR, 'val',   YOLO_DIR)
print('YOLO dataset ready')

Train images: 837 | Val images: 148
[train] Done. Skipped 0 images not found on disk.
[val] Done. Skipped 0 images not found on disk.
YOLO dataset ready


## 9. Create data.yaml

In [9]:
data_yaml_content = {
    'path': YOLO_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 3,
    'names': CLASS_NAMES
}
yaml_path = f'{YOLO_DIR}/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)
print('data.yaml contents:')
print(open(yaml_path).read())

data.yaml contents:
names:
- Plant_bean
- Flower_open
- Flower_closed
nc: 3
path: /kaggle/working/yolo_dataset
train: images/train
val: images/val



## 10. Train YOLOv8x-seg (in segments)
> **How to use:** Run with `START_EPOCH = 0` first. After each segment finishes, download the checkpoint from the Output tab, then increase `START_EPOCH` by `SEGMENT_SIZE` and run again.

In [ ]:
START_EPOCH  = 0    # ← change to 10, 20, 30... to resume
SEGMENT_SIZE = 10   # epochs per segment

if START_EPOCH == 0:
    model = YOLO('yolov8x-seg.pt')
else:
    model = YOLO(f'/kaggle/working/best_epoch_{START_EPOCH}.pt')
    print(f"Resuming from epoch {START_EPOCH}")

device = 0 if torch.cuda.is_available() else 'cpu'
print(f'Using device: {"GPU" if device == 0 else "CPU"}')

results = model.train(
    data          = yaml_path,
    epochs        = START_EPOCH + SEGMENT_SIZE,
    imgsz         = 1024,
    batch         = 4,
    patience      = 999,        
    device        = device,
    project       = RUNS_DIR,
    name          = 'bean_seg',
    exist_ok      = True,
    resume        = START_EPOCH > 0,
    optimizer     = 'AdamW',
    lr0           = 0.001,
    lrf           = 0.01,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    flipud        = 0.5,
    fliplr        = 0.5,
    mosaic        = 1.0,
    degrees       = 15.0,
    translate     = 0.1,
    scale         = 0.5,
    copy_paste    = 0.3,
    mixup         = 0.15,
)

checkpoint_name = f'best_epoch_{START_EPOCH + SEGMENT_SIZE}.pt'
src = f'{RUNS_DIR}/bean_seg/weights/best.pt'
shutil.copy(src, f'/kaggle/working/{checkpoint_name}')
shutil.copy(src, BEST_MODEL)  # always keep best.pt up to date too

print(f"\n Segment done! Epochs {START_EPOCH} → {START_EPOCH + SEGMENT_SIZE}")
print(f"Saved: /kaggle/working/{checkpoint_name}")
print(f"Next:  set START_EPOCH = {START_EPOCH + SEGMENT_SIZE} and run this cell again")

Using device: GPU
Ultralytics 8.4.31 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolov8x-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=bean_seg, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_ma

## ✅ CHECKPOINT — Download Model Weights Now!

In [ ]:
# Run this immediately after training finishes — download before anything else!
import os, shutil
from IPython.display import FileLink, display

# Make sure best.pt is in working dir
if not os.path.exists(BEST_MODEL):
    src = f'{RUNS_DIR}/bean_seg/weights/best.pt'
    shutil.copy(src, BEST_MODEL)

size_mb = os.path.getsize(BEST_MODEL) / 1024**2
print(f"best.pt → {size_mb:.1f} MB")
print("Go to: Right sidebar → Output tab → best.pt → click download icon")

## 11. Validate Model

In [ ]:
model = YOLO(BEST_MODEL)

metrics = model.val(data=yaml_path, imgsz=1024, iou=0.50, conf=0.25)
print("\n===== Validation Metrics =====")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

## 12. Find Best Confidence Threshold (optional but recommended)

In [ ]:
# Uses bbox from points column since your CSV uses polygon points, not bbox
def get_bbox_from_points(points_str, img_w, img_h):
    try:
        points = parse_points(points_str)
        xs = [p[0] for p in points]
        ys = [p[1] for p in points]
        return {'ymin': min(ys), 'xmin': min(xs), 'ymax': max(ys), 'xmax': max(xs)}
    except:
        return None

def get_test_image_path(img_id, img_dir):
    for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
        p = Path(img_dir) / f"{img_id}{ext}"
        if p.exists():
            return str(p)
    return None

def compute_iou_bbox(boxA, boxB):
    yA = max(boxA[0], boxB[0]); xA = max(boxA[1], boxB[1])
    yB = min(boxA[2], boxB[2]); xB = min(boxA[3], boxB[3])
    inter = max(0, yB - yA) * max(0, xB - xA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0

def f1_at_iou50(preds, gts, iou_thresh=0.50):
    tp = fp = 0
    matched = set()
    for pred in preds:
        best_iou, best_idx = 0, -1
        for idx, gt in enumerate(gts):
            if idx in matched or pred['class'] != gt['class']:
                continue
            iou = compute_iou_bbox(
                [pred['ymin'], pred['xmin'], pred['ymax'], pred['xmax']],
                [gt['ymin'],   gt['xmin'],   gt['ymax'],   gt['xmax']]
            )
            if iou > best_iou:
                best_iou, best_idx = iou, idx
        if best_iou >= iou_thresh:
            tp += 1; matched.add(best_idx)
        else:
            fp += 1
    fn = len(gts) - len(matched)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return f1, precision, recall

# Build ground truth from polygon points
gt_by_image = defaultdict(list)
for _, row in val_split.iterrows():
    bbox = get_bbox_from_points(row['points'], row['image_width'], row['image_height'])
    if bbox:
        bbox['class'] = row['class']
        gt_by_image[row['Image_ID']].append(bbox)

IOU_THRESHOLD = 0.45
thresholds    = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
sweep_results = []

print("Sweeping confidence thresholds on 50 val images...")
for conf_thresh in thresholds:
    all_preds, all_gts = [], []
    for img_id in list(gt_by_image.keys())[:50]:
        img_path = get_test_image_path(img_id, TRAIN_IMG_DIR)
        if img_path is None:
            continue
        pred_result = model.predict(
            source=img_path, imgsz=1024,
            conf=conf_thresh, iou=IOU_THRESHOLD,
            augment=True, verbose=False
        )[0]
        preds = []
        if pred_result.boxes is not None:
            for j in range(len(pred_result.boxes)):
                x1,y1,x2,y2 = pred_result.boxes.xyxy[j].cpu().numpy()
                preds.append({
                    'class': CLASS_NAMES[int(pred_result.boxes.cls[j])],
                    'ymin': float(y1), 'xmin': float(x1),
                    'ymax': float(y2), 'xmax': float(x2)
                })
        all_preds.extend(preds)
        all_gts.extend(gt_by_image[img_id])
    f1, prec, rec = f1_at_iou50(all_preds, all_gts)
    sweep_results.append({'conf': conf_thresh, 'F1': f1, 'Precision': prec, 'Recall': rec})
    print(f"  conf={conf_thresh:.2f}  F1={f1:.4f}  Prec={prec:.4f}  Rec={rec:.4f}")

sweep_df = pd.DataFrame(sweep_results)
best_row = sweep_df.loc[sweep_df['F1'].idxmax()]
BEST_CONF = best_row['conf']
print(f"\n🏆 Best threshold: {BEST_CONF} → F1={best_row['F1']:.4f}")
print(f"Using CONF_THRESHOLD = {BEST_CONF} for final inference")

## 13. Run Inference on Test Set

In [ ]:
# Use best conf from sweep above, or set manually if you skipped sweep
CONF_THRESHOLD = BEST_CONF if 'BEST_CONF' in dir() else 0.25
IOU_THRESHOLD  = 0.45

print(f"Using CONF_THRESHOLD={CONF_THRESHOLD}  IOU_THRESHOLD={IOU_THRESHOLD}")
print(f"Model: {BEST_MODEL}")

model = YOLO(BEST_MODEL)

submission_rows = []
test_ids = test_df["Image_ID"].unique()
missing  = []

print(f"Running inference on {len(test_ids)} test images...")

for i, img_id in enumerate(test_ids):
    img_path = get_test_image_path(img_id, TEST_IMG_DIR)

    if img_path is None:
        missing.append(img_id)
        submission_rows.append({
            'Image_ID': img_id, 'class': '__background__',
            'confidence': 0.0, 'ymin': 0, 'xmin': 0, 'ymax': 0, 'xmax': 0
        })
        continue

    results = model.predict(
        source  = img_path,
        imgsz   = 1024,       # matches training size
        conf    = CONF_THRESHOLD,
        iou     = IOU_THRESHOLD,
        augment = True,       # test-time augmentation
        verbose = False
    )
    result = results[0]
    boxes  = result.boxes

    if boxes is None or len(boxes) == 0:
        submission_rows.append({
            'Image_ID': img_id, 'class': '__background__',
            'confidence': 0.0, 'ymin': 0, 'xmin': 0, 'ymax': 0, 'xmax': 0
        })
        continue

    orig_h, orig_w = result.orig_shape
    for j in range(len(boxes)):
        conf     = float(boxes.conf[j].cpu())
        cls_id   = int(boxes.cls[j].cpu())
        cls_name = CLASS_NAMES[cls_id]

        if result.masks is not None and j < len(result.masks):
            mask = result.masks.data[j].cpu().numpy()
            mask_resized = cv2.resize(mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
            ys, xs = np.where(mask_resized > 0.5)
            if len(xs) > 0 and len(ys) > 0:
                xmin, xmax = int(xs.min()), int(xs.max())
                ymin, ymax = int(ys.min()), int(ys.max())
            else:
                x1,y1,x2,y2 = boxes.xyxy[j].cpu().numpy()
                xmin,ymin,xmax,ymax = int(x1),int(y1),int(x2),int(y2)
        else:
            x1,y1,x2,y2 = boxes.xyxy[j].cpu().numpy()
            xmin,ymin,xmax,ymax = int(x1),int(y1),int(x2),int(y2)

        submission_rows.append({
            'Image_ID': img_id, 'class': cls_name,
            'confidence': round(conf, 6),
            'ymin': ymin, 'xmin': xmin, 'ymax': ymax, 'xmax': xmax
        })

    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(test_ids)} processed...")

print(f"\nInference done Missing images: {len(missing)}")

## 14. Build & Save Submission CSV

In [ ]:
submission_df = pd.DataFrame(submission_rows, columns=[
    'Image_ID', 'class', 'confidence', 'ymin', 'xmin', 'ymax', 'xmax'
])

# Sort by image then confidence descending
submission_df = submission_df.sort_values(
    ['Image_ID', 'confidence'], ascending=[True, False]
)

# Remove exact duplicate detections
submission_df = submission_df.drop_duplicates(
    subset=['Image_ID', 'class', 'ymin', 'xmin', 'ymax', 'xmax']
).reset_index(drop=True)

# Cover any missing test images
submitted_ids = set(submission_df['Image_ID'].unique())
missing_ids   = set(test_ids) - submitted_ids
if missing_ids:
    print(f"Adding background rows for {len(missing_ids)} missing images")
    bg_rows = [{
        'Image_ID': img_id, 'class': '__background__',
        'confidence': 0.0, 'ymin': 0, 'xmin': 0, 'ymax': 0, 'xmax': 0
    } for img_id in missing_ids]
    submission_df = pd.concat([submission_df, pd.DataFrame(bg_rows)], ignore_index=True)
else:
    print("All test images covered!")

submission_df.to_csv(SUBMISSION, index=False)

print(f"\nSubmission Summary")
print(f"  Total rows:      {len(submission_df)}")
print(f"  Unique images:   {submission_df['Image_ID'].nunique()}")
print(f"  Expected images: {len(test_ids)}")
print()
print("Class distribution:")
print(submission_df['class'].value_counts())
print()
display(submission_df.head(10))

## ✅ CHECKPOINT — Download Submission CSV Now!

In [ ]:
import os, shutil

size_kb = os.path.getsize(SUBMISSION) / 1024
print(f"submission.csv → {size_kb:.1f} KB  |  {len(submission_df)} rows")

# Copy so it appears clearly in Output tab
shutil.copy(SUBMISSION, '/kaggle/working/submission_final.csv')
print("\nGo to: Right sidebar → Output tab → submission_final.csv → download icon")
print("OR paste this path when submitting on the competition page: /kaggle/working/submission.csv")